<a href="https://colab.research.google.com/github/CienciaDatosUdea/005_CCA_Estudiantes/blob/main/Sesiones/U4_S1_Grafos_ML.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Unidad 4 · Sesión 1 — Grafos: fundamentos y ML tradicional

Cuaderno de ilustración de la clase. Sigue el orden de las diapositivas *GNN foundations and molecular representation*:

- **Parte I (frames 3–8):** un sistema físico como grafo con atributos.
- **Parte II (frames 10–15):** ML tradicional sobre grafos, sin aprendizaje profundo.

Lo ejecutamos juntos celda por celda. Hay tres **ejercicios** para resolver en clase; su solución está en una celda oculta justo debajo (en Colab, haga clic en *Mostrar código*).

In [ ]:
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
from pathlib import Path
from urllib.request import urlretrieve
from scipy.sparse import identity
from scipy.sparse.linalg import eigsh, splu
from sklearn.metrics import roc_auc_score

# Parte I — Sistema físico como grafo con atributos

## 1. Teoría de grafos: los puentes de Königsberg (frame 3)

Cada región de tierra es un nodo y cada puente una arista. Hay regiones unidas por dos puentes, así que usamos un `MultiGraph`, que admite aristas repetidas.

In [ ]:
K = nx.MultiGraph()
K.add_edges_from([("A", "B"), ("A", "B"),   # isla - orilla norte
                  ("A", "C"), ("A", "C"),   # isla - orilla sur
                  ("A", "D"), ("B", "D"), ("C", "D")])
pos_K = {"A": (0, 0), "B": (0, 1), "C": (0, -1), "D": (1.5, 0)}

Para ver los puentes repetidos, cada copia se dibuja con una curvatura distinta (`connectionstyle`).

In [ ]:
nx.draw(K, pos_K, with_labels=True, node_color="lightblue", node_size=800,
        arrows=True, arrowstyle="-", connectionstyle=["arc3,rad=0", "arc3,rad=0.3"])
plt.show()

**Teorema de Euler:** un grafo conexo tiene un ciclo euleriano (recorre cada arista una vez y vuelve al inicio) si y solo si todo nodo tiene grado par.

In [ ]:
print("Grados:", dict(K.degree()))
print("¿Todos pares?", all(d % 2 == 0 for _, d in K.degree()))
print("¿Ciclo euleriano según NetworkX?", nx.is_eulerian(K))

> **Ejercicio 1.** Quite un puente entre `A` y `B`. ¿Existe ahora un ciclo euleriano? ¿Existe un **camino** euleriano, que recorra cada puente una vez pero no necesita cerrarse como un ciclo? La condición para un camino es que haya exactamente 0 o 2 nodos de grado impar. Resuelvalo en una hoja antes de mirar la solucion.

In [ ]:
#@title Solución del ejercicio 1
K1 = K.copy()
K1.remove_edge("A", "B")
print("¿Ciclo euleriano según NetworkX?", nx.is_eulerian(K))
impares = [v for v, d in K1.degree() if d % 2 == 1]
print("Nodos de grado impar:", impares)                 # C y D: exactamente dos
print("¿Camino euleriano?", nx.has_eulerian_path(K1))
print("Un camino:", list(nx.eulerian_path(K1)))         # empieza y termina en los nodos impares

## 2. El modelo de Ising como problema de grafos (frame 4)

Cada nodo es un espín $\sigma_i=\pm1$ y cada arista un acoplamiento: $H=\sum_{(i,j)\in E}J_{ij}\,\sigma_i\sigma_j$. Para el ferromagneto $J_{ij}=-J$, y con la matriz de adyacencia
$$
H(\boldsymbol\sigma)=-J\sum_{(i,j)\in E}\sigma_i\sigma_j=-\frac{J}{2}\,\boldsymbol\sigma^{\top}A\,\boldsymbol\sigma ,
$$
porque $\boldsymbol\sigma^{\top}A\boldsymbol\sigma$ cuenta cada arista dos veces.

Primero miramos una red cuadrada periódica pequeña, de $4\times4$.

In [ ]:
red = nx.grid_2d_graph(4, 4, periodic=True)
pos_red = {(i, j): (j, -i) for i, j in red}

In [ ]:
nx.draw(red, pos_red, node_color="lightblue", node_size=300,
        arrows=True, arrowstyle="-", connectionstyle="arc3,rad=0.15")
plt.show()

Las aristas curvas largas unen la primera y la última fila (y columna): son las condiciones periódicas, y por eso todos los nodos tienen grado 4.

Ahora cargamos las configuraciones $16\times16$ del curso y calculamos la energía con la matriz de adyacencia. El orden de los nodos debe coincidir con el de `reshape(-1)`, fila por fila.

In [ ]:
url = "https://github.com/CienciaDatosUdea/005_CCA_Estudiantes/raw/main/Laboratorios/Datos/ising_dataset.npz"
if not Path("ising_dataset.npz").exists():
    urlretrieve(url, "ising_dataset.npz")
datos = np.load("ising_dataset.npz")
X, T = datos["X"].astype(int), datos["T"]
print(X.shape, "configuraciones a las temperaturas", np.unique(T))

In [ ]:
L = 16
orden = [(i, j) for i in range(L) for j in range(L)]          # fila por fila
A16 = nx.to_numpy_array(nx.grid_2d_graph(L, L, periodic=True), nodelist=orden)

E = np.array([-0.5 * s.ravel() @ A16 @ s.ravel() for s in X])
E_roll = np.array([-np.sum(s * (np.roll(s, 1, 0) + np.roll(s, 1, 1))) for s in X])
print("¿Coincide con el cálculo con np.roll?", np.allclose(E, E_roll))

La energía por espín sube con la temperatura y cambia de régimen cerca de $T_c$ de Onsager.

In [ ]:
temps = np.unique(T)
plt.plot(temps, [E[T == t].mean() / L**2 for t in temps], "o-")
plt.axvline(2.269, ls="--", color="gray")
plt.xlabel("T"); plt.ylabel("E / N")
plt.show()

## 3. ¿Cuánto cuesta la función de partición? (frames 4–5)

$Z(T)=\sum_{\boldsymbol\sigma}e^{-H(\boldsymbol\sigma)/T}$ suma sobre las $2^N$ configuraciones. La forma más directa es enumerarlas todas: la configuración número $c$ se lee en los bits de $c$.

In [ ]:
def Z_exacta(G, T):
    G = nx.convert_node_labels_to_integers(G)
    N = G.number_of_nodes()
    s = 1 - 2 * ((np.arange(2**N)[:, None] >> np.arange(N)) & 1)   # todas las configuraciones
    i, j = np.array(G.edges()).T
    H = -(s[:, i] * s[:, j]).sum(axis=1)
    return np.exp(-H / T).sum()

In [ ]:
import time
N_lista, tiempos = [], []
for Lx, Ly in [(3, 3), (3, 4), (4, 4), (4, 5)]:
    t0 = time.time()
    Z_exacta(nx.grid_2d_graph(Lx, Ly, periodic=True), T=2.5)
    N_lista.append(Lx * Ly); tiempos.append(time.time() - t0)
print("N:", N_lista, " tiempo (s):", np.round(tiempos, 3))

En escala logarítmica el tiempo crece como una recta: el costo es exponencial en $N$. Para la red de $16\times16$ habría que sumar $2^{256}\approx10^{77}$ términos. Por eso importa saber en qué clase de complejidad está el problema (frame 5).

In [ ]:
plt.semilogy(N_lista, tiempos, "o-")
plt.xlabel("número de espines N"); plt.ylabel("tiempo (s)")
plt.show()

In [ ]:
# Red tres dimensional
N_3d, tiempos_3d = [], []
for dims in [(2, 2, 2), (2, 2, 3), (2, 2, 4), (2, 3, 3), (2, 2, 5)]:
    red3 = nx.grid_graph(dim=dims, periodic=True)       # red cúbica periódica
    t0 = time.time()
    Z = Z_exacta(red3, T=4.5)
    N_3d.append(red3.number_of_nodes()); tiempos_3d.append(time.time() - t0)
    print(dims, "N =", N_3d[-1], " aristas =", red3.number_of_edges(),
          " ¿plano?", nx.check_planarity(red3)[0])
print("N:", N_3d, " tiempo (s):", np.round(tiempos_3d, 3))

In [ ]:
plt.semilogy(N_lista, tiempos, "o-", label="red 2D")
plt.semilogy(N_3d, tiempos_3d, "s-", label="red 3D")
plt.xlabel("número de espines N"); plt.ylabel("tiempo (s)"); plt.legend()
plt.show()

En ambos casos red 2D y la red 3D el costo de calcular la funcion de partición completa es exponencial. ¿Que significa que un problema sea entonces P y el otro NP? ¿Que propiedad del grafo cambia cuando pasa de plano a no plano? Si el tiempo lo permite: discusión en clase de grafos, embeddings, topología.

## 4. Una molécula como grafo multirrelacional (frames 6–7)

El 4-formilbenzonitrilo, C$_8$H$_5$NO, tiene 15 átomos. Cada tipo de enlace es una **relación** $\tau$ y hay una matriz $A_\tau$ por relación, apiladas en el tensor $A\in\{0,1\}^{|\mathcal V|\times|\mathcal R|\times|\mathcal V|}$.

In [ ]:
enlaces = [("C1", "C2", "aromático"), ("C2", "C3", "aromático"), ("C3", "C4", "aromático"),
           ("C4", "C5", "aromático"), ("C5", "C6", "aromático"), ("C6", "C1", "aromático"),
           ("C2", "H2", "simple"), ("C3", "H3", "simple"), ("C5", "H5", "simple"),
           ("C6", "H6", "simple"), ("C1", "C7", "simple"), ("C4", "C8", "simple"),
           ("C8", "H8", "simple"), ("C8", "O", "doble"), ("C7", "N", "triple")]
mol = nx.Graph()
for u, v, r in enlaces:
    mol.add_edge(u, v, relacion=r)

In [ ]:
color = {"C": "gray", "H": "white", "N": "royalblue", "O": "tomato"}
estilo = {"aromático": "dashed", "simple": "solid", "doble": "solid", "triple": "solid"}
ancho = {"aromático": 2, "simple": 1, "doble": 3, "triple": 5}
rel = [mol.edges[e]["relacion"] for e in mol.edges]
nx.draw(mol, nx.kamada_kawai_layout(mol), with_labels=True, font_size=8,
        node_color=[color[n[0]] for n in mol], edgecolors="black",
        style=[estilo[r] for r in rel], width=[ancho[r] for r in rel])
plt.show()

El tensor se llena relación por relación. La matriz de atributos $X$ tiene una fila por átomo, en el mismo orden: aquí, su número atómico $Z$.

In [ ]:
atomos = list(mol.nodes())
relaciones = ["aromático", "simple", "doble", "triple"]
A = np.zeros((len(atomos), len(relaciones), len(atomos)), dtype=int)
for u, v, r in enlaces:
    i, k, j = atomos.index(u), relaciones.index(r), atomos.index(v)
    A[i, k, j] = A[j, k, i] = 1

Z = {"C": 6, "H": 1, "N": 7, "O": 8}
X_mol = np.array([[Z[a[0]]] for a in atomos])
print("Tensor:", A.shape, " aristas por relación:", A.sum(axis=(0, 2)) // 2)

**Tareas de aprendizaje (frame 8).** Con esta representación se pueden plantear tareas de aprendizaje: clasificar nodos (p. ej. el tipo de átomo), predecir relaciones (un enlace faltante), detectar comunidades (grupos funcionales) y predecir una propiedad del grafo completo, como la toxicidad de la molécula.

# Parte II — ML tradicional sobre grafos

## 5. El poder de la familia Médici (frame 10)

Red de matrimonios entre familias florentinas del siglo XV (Padgett y Ansell, 1993; Hamilton, fig. 2.1). Viene incluida en NetworkX.

In [ ]:
F = nx.florentine_families_graph()
familias = list(F.nodes())
A_F = nx.to_numpy_array(F, nodelist=familias)
pos_F = nx.spring_layout(F, seed=3)

In [ ]:
nx.draw(F, pos_F, with_labels=True, node_color="lightblue", font_size=8)
plt.show()

## 6. Propiedades de nodos para clasificación de nodos (frame 11)

- **Grado:** $d_u=\sum_vA[u,v]$, la suma de cada fila.
- **Centralidad de vector propio:** $\lambda\mathbf e=A\mathbf e$. Se obtiene repitiendo $\mathbf e\leftarrow A\mathbf e/\|A\mathbf e\|$ (iteración de potencias).
- **Agrupamiento:** fracción de pares de vecinos conectados. Las aristas entre vecinos de $u$ son $(A^3)_{uu}/2$, los triángulos que pasan por $u$.

In [ ]:
d = A_F.sum(axis=1)

e = np.ones(len(familias))
for _ in range(100):
    e = A_F @ e / np.linalg.norm(A_F @ e)

triangulos = np.diag(A_F @ A_F @ A_F) / 2
pares = d * (d - 1) / 2
c = np.zeros_like(d)
c[pares > 0] = triangulos[pares > 0] / pares[pares > 0]

for k in np.argsort(-e)[:6]:
    print(f"{familias[k]:<12} grado={d[k]:.0f}  centralidad={e[k]:.2f}  agrupamiento={c[k]:.2f}")

Los Médici tienen el mayor grado y la mayor centralidad. En la figura, el tamaño es la centralidad y el color el agrupamiento.

In [ ]:
nx.draw(F, pos_F, nodelist=familias, with_labels=True, font_size=8,
        node_size=2500 * e, node_color=c, cmap="viridis")
plt.show()

In [ ]:
#Usando la libreria Networkx
grado_nx = [F.degree(f) for f in familias]
centralidad_nx = [nx.eigenvector_centrality_numpy(F)[f] for f in familias]
agrupamiento_nx = [nx.clustering(F)[f] for f in familias]

print("¿Grado igual?       ", np.allclose(d, grado_nx))
print("¿Centralidad igual? ", np.allclose(e, centralidad_nx))
print("¿Agrupamiento igual?", np.allclose(c, agrupamiento_nx))

## 7. Propiedades de nodos por agregación (frame 12)

Cada ronda suma la propiedad de los vecinos, $h^{(k)}(u)=\sum_{v\in\mathcal N(u)}h^{(k-1)}(v)$, que en forma matricial es $\mathbf h^{(k)}=A^k\mathbf h^{(0)}$. Tomamos la cadena $a$–$b$–$c$–$d$ con $h^{(0)}=d_u$.

In [ ]:
cadena = nx.path_graph(["a", "b", "c", "d"])
A_c = nx.to_numpy_array(cadena) #calcula matriz de adyacencia
h0 = A_c.sum(axis=1) #grado del nodo: suma de los vecinos
print("h0 =", h0, " h1 =", A_c @ h0, " h2 =", A_c @ A_c @ h0)

$h^{(2)}(a)=3=d_a+d_c$: tras dos rondas, el nodo $a$ ya incorpora al nodo $c$, que está a dos saltos.

## 8. El algoritmo de Weisfeiler–Lehman (frame 13)

En cada ronda, la etiqueta nueva de $v$ es un HASH de su etiqueta y del multiconjunto de las de sus vecinos. Implementamos el HASH con un diccionario que asigna un entero nuevo a cada par distinto, compartido entre los grafos que se comparan.

In [ ]:
def wl(grafos, K=3):
    tabla = {}
    etiquetas = [{v: G.degree(v) for v in G} for G in grafos]
    historia = [[sorted(l.values())] for l in etiquetas]
    for _ in range(K):
        for G, l, h in zip(grafos, etiquetas, historia):
            pares = {v: (l[v], tuple(sorted(l[u] for u in G[v]))) for v in G}
            l.update({v: tabla.setdefault(p, len(tabla)) for v, p in pares.items()})
            h.append(sorted(l.values()))
    return historia          # por grafo: etiquetas de cada ronda (el histograma)

Dos grafos isomorfos tienen el mismo histograma. El recíproco falla: dos triángulos y un hexágono no son isomorfos, pero 1-WL no los distingue.

In [ ]:
triangulos2 = nx.disjoint_union(nx.cycle_graph(3), nx.cycle_graph(3))
hexagono = nx.cycle_graph(6)

In [ ]:
pos_t = {k: (2.5 * (k // 3) + np.cos(2 * np.pi * k / 3), np.sin(2 * np.pi * k / 3)) for k in range(6)}
fig, ax = plt.subplots(1, 2, figsize=(7, 3))
nx.draw(triangulos2, pos_t, ax=ax[0], node_color="lightblue")
nx.draw(hexagono, nx.circular_layout(hexagono), ax=ax[1], node_color="lightblue")
plt.show()

In [ ]:
h_t, h_h = wl([triangulos2, hexagono])
print("¿Isomorfos?", nx.is_isomorphic(triangulos2, hexagono))
print("¿Mismo histograma 1-WL?", h_t == h_h)

> **Ejercicio 2.** La decalina (dos anillos de 6 carbonos fusionados) y el biciclopentilo (dos anillos de 5 unidos por un enlace) tienen 10 carbonos. Construya sus grafos de carbonos, dibújelos y compruebe si 1-WL los distingue.

In [ ]:
#@title Solución del ejercicio 2
decalina = nx.cycle_graph(10)
decalina.add_edge(0, 5)                       # el puente que fusiona los dos anillos de 6
biciclopentilo = nx.disjoint_union(nx.cycle_graph(5), nx.cycle_graph(5))
biciclopentilo.add_edge(0, 5)                 # el enlace que une los dos anillos de 5

fig, ax = plt.subplots(1, 2, figsize=(7, 3))
nx.draw(decalina, nx.kamada_kawai_layout(decalina), ax=ax[0], node_color="gray")
nx.draw(biciclopentilo, nx.kamada_kawai_layout(biciclopentilo), ax=ax[1], node_color="gray")
plt.show()

h_d, h_b = wl([decalina, biciclopentilo])
print("¿Isomorfos?", nx.is_isomorphic(decalina, biciclopentilo))
print("¿Mismo histograma 1-WL?", h_d == h_b)  # True: 1-WL no ve el tamaño de los anillos

### 1-WL para encontrar átomos equivalentes en una molécula

Si dos átomos ocupan la misma posición en el grafo molecular, 1-WL les asigna la misma etiqueta final. En química, esos átomos son **equivalentes**: por ejemplo, dan una sola señal en un espectro de resonancia magnética nuclear (RMN). Aplicamos 1-WL al 4-formilbenzonitrilo de la sección 4 con dos etiquetas iniciales: el grado (solo la topología) y el elemento químico (un atributo de nodo).

In [ ]:
def clases_wl(G, etiqueta, rondas=5):
    l = dict(etiqueta)
    for _ in range(rondas):
        tabla = {}
        l = {v: tabla.setdefault((l[v], tuple(sorted(l[u] for u in G[v]))), len(tabla)) for v in G}
    clases = {}
    for v, x in l.items():
        clases.setdefault(x, []).append(v)
    return sorted(sorted(c) for c in clases.values())

por_grado = clases_wl(mol, {v: mol.degree(v) for v in mol},1)
por_elemento = clases_wl(mol, {v: v[0] for v in mol},1)       # "C", "H", "N" u "O"
print("Con el grado:   ", [c for c in por_grado if len(c) > 1])
print("Con el elemento:", [c for c in por_elemento if len(c) > 1])

Cambia el número de rondas a llegar a la estabilidad

## 9. Atributos locales para predicción de relación (frame 14)

Usamos la red [ca-GrQc de SNAP](https://snap.stanford.edu/data/ca-GrQc.html): dos autores están conectados si publicaron juntos en arXiv un artículo de relatividad general y cosmología cuántica. Trabajamos con su componente conexa más grande.

In [ ]:
if not Path("ca-GrQc.txt.gz").exists():
    urlretrieve("https://snap.stanford.edu/data/ca-GrQc.txt.gz", "ca-GrQc.txt.gz")
G = nx.read_edgelist("ca-GrQc.txt.gz", nodetype=int)
G.remove_edges_from(nx.selfloop_edges(G))
G = nx.convert_node_labels_to_integers(G.subgraph(max(nx.connected_components(G), key=len)))
n = G.number_of_nodes()
print(n, "autores,", G.number_of_edges(), "colaboraciones")

Para probar una predicción, **ocultamos** 500 aristas (los positivos) y tomamos 500 pares de autores que nunca colaboraron (los negativos). Las similitudes se calculan con el grafo sin las aristas ocultas.

In [ ]:
rng = np.random.default_rng(0)
aristas = np.array(G.edges())
ocultas = aristas[rng.choice(len(aristas), 500, replace=False)]
G_train = G.copy()
G_train.remove_edges_from(ocultas)

negativos = []
while len(negativos) < 500:
    u, v = rng.integers(n, size=2)
    if u != v and not G.has_edge(u, v):
        negativos.append((u, v))

pares = np.vstack([ocultas, negativos])
y = np.r_[np.ones(500), np.zeros(500)]          # 1 = arista oculta, 0 = no existe
A = nx.to_scipy_sparse_array(G_train, nodelist=range(n), format="csr", dtype=float)

La similitud más simple cuenta los vecinos comunes, $S=A^2$. Basta multiplicar las filas $u$ y $v$ de $A$ para cada par. La de Katz suma los caminos de todas las longitudes,
$$
S_{\mathrm{Katz}}=\sum_{i\ge1}\beta^iA^i=(I-\beta A)^{-1}-I,\qquad \beta<1/\lambda_1 .
$$

**Forma operativa.** No invertimos $I-\beta A$: aunque $A$ es dispersa (cada autor tiene pocos coautores), su inversa es densa, con unos $4158^2\approx1.7\times10^7$ entradas, y además solo necesitamos $S_{\mathrm{Katz}}[u,v]$ para los 1000 pares de prueba. El cálculo se hace en cuatro pasos:

1. **Una columna es un sistema lineal.** Si $\mathbf e_v$ es el vector con un 1 en la posición $v$ y ceros en el resto, la columna $v$ de $(I-\beta A)^{-1}$ es la solución $\mathbf s_v$ de
$$
(I-\beta A)\,\mathbf s_v=\mathbf e_v .
$$
Restando $\mathbf e_v$ (la columna $v$ de la identidad) se obtiene la columna $v$ de $S_{\mathrm{Katz}}$, y su componente $u$ es la similitud $S_{\mathrm{Katz}}[u,v]$.
2. **Se factoriza una sola vez.** Todos los sistemas tienen la misma matriz $I-\beta A$. La factorización LU la escribe como $I-\beta A=LU$, con $L$ triangular inferior y $U$ triangular superior. Es el paso caro, pero se hace una vez; como $A$ es dispersa, `splu` produce factores también dispersos.
3. **Solo las columnas necesarias.** Juntamos los nodos $v$ que aparecen como segundo elemento de algún par de prueba y armamos una matriz $E$ cuyas columnas son sus vectores $\mathbf e_v$.
4. **Resolver y leer.** Con $L$ y $U$, cada sistema se resuelve con dos sustituciones baratas, hacia adelante y hacia atrás. `lu.solve(E)` resuelve todas las columnas a la vez; restamos $E$ y, para cada par $(u,v)$, tomamos la componente $u$ de la columna de $v$.


In [ ]:
S_cn = np.asarray(A[pares[:, 0]].multiply(A[pares[:, 1]]).sum(axis=1)).ravel()

lam1 = eigsh(A, k=1, which="LA")[0][0]
beta = 0.5 / lam1
lu = splu((identity(n) - beta * A).tocsc())            # una sola factorización

columnas = np.unique(pares[:, 1])
E = np.zeros((n, len(columnas)))
E[columnas, np.arange(len(columnas))] = 1              # vectores e_v
S_columnas = lu.solve(E) - E                           # s_v = (I - beta A)^{-1} e_v - e_v
S_katz = S_columnas[pares[:, 0], np.searchsorted(columnas, pares[:, 1])]
print(f"lambda_1 = {lam1:.2f}, beta = {beta:.4f}")

La probabilidad de una arista se toma proporcional a $S[u,v]$, así que la similitud sirve de puntaje. El AUC mide qué tan bien ordena las aristas ocultas por encima de los negativos: 0,5 es azar y 1 es perfecto.

In [ ]:
print("AUC vecinos comunes:", round(roc_auc_score(y, S_cn), 3))
print("AUC Katz:           ", round(roc_auc_score(y, S_katz), 3))
print("Pares con 0 vecinos comunes:", np.mean(S_cn == 0))

Muchos pares no tienen vecinos comunes, así que con esa similitud quedan empatados en cero. Katz sí los distingue, porque cuenta caminos más largos. El histograma muestra cómo Katz separa los positivos de los negativos.

In [ ]:
plt.hist(np.log10(S_katz[y == 1] + 1e-12), bins=30, alpha=0.6, label="aristas ocultas")
plt.hist(np.log10(S_katz[y == 0] + 1e-12), bins=30, alpha=0.6, label="pares sin arista")
plt.xlabel("log10 S_Katz"); plt.ylabel("número de pares"); plt.legend()
plt.savefig("histograma_katz.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
plt.hist(np.log10(S_cn[y == 1] + 1e-12), bins=30, alpha=0.6, label="aristas ocultas")
plt.hist(np.log10(S_cn[y == 0] + 1e-12), bins=30, alpha=0.6, label="pares sin arista")
plt.xlabel("log10 S_cn"); plt.ylabel("número de pares"); plt.legend()
plt.savefig("histograma_cn.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
from google.colab import files
files.download("histograma_katz.png")
files.download("histograma_cn.png")

Las dos similitudes separan bien porque los positivos se concentran a la derecha y los negativos a la izquierda. Katz mejora a vecinos comunes en un punto concreto: ordena los pares que vecinos comunes deja empatados en cero. Unas 15 de las 50 aristas ocultas sin vecinos comunes suben a las bandas intermedias, entre −9 y −5, por encima de la mayoría de los negativos. Los dos AUC son altos y cercanos.

## 10. Laplaciano del grafo para detección de comunidades (frame 15)

Con $L=D-A$, el signo del autovector del segundo menor autovalor (vector de Fiedler) parte el grafo en dos grupos que aproximan el mínimo de
$$
\operatorname{RatioCut}(\mathcal A,\bar{\mathcal A})=\operatorname{cut}(\mathcal A,\bar{\mathcal A})\left(\tfrac{1}{|\mathcal A|}+\tfrac{1}{|\bar{\mathcal A}|}\right).
$$

In [ ]:
L_F = np.diag(d) - A_F
valores, vectores = np.linalg.eigh(L_F)          # autovalores en orden ascendente
grupo_fiedler = {familias[k] for k in range(len(familias)) if vectores[k, 1] >= 0}

def ratiocut(G, grupo):
    corte = sum((u in grupo) != (v in grupo) for u, v in G.edges())
    return corte * (1 / len(grupo) + 1 / (len(G) - len(grupo)))

print("RatioCut con el vector de Fiedler:", round(ratiocut(F, grupo_fiedler), 3))

Con 15 nodos podemos encontrar el mínimo exacto revisando las $2^{14}$ particiones (fijamos la primera familia en $\mathcal A$).

In [ ]:
mejor = min((ratiocut(F, {familias[0]} | {familias[k + 1] for k in range(14) if m >> k & 1}), m)
            for m in range(2**14 - 1))
grupo_exacto = {familias[0]} | {familias[k + 1] for k in range(14) if mejor[1] >> k & 1}
print("RatioCut mínimo exacto:", round(mejor[0], 3))

El vector de Fiedler resuelve una versión relajada del problema, así que su partición no siempre es la óptima. El mínimo exacto exige revisar $2^{|\mathcal V|-1}$ particiones: el mismo costo exponencial de la función de partición de Ising.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10, 4))
for a, grupo, titulo in [(ax[0], grupo_fiedler, "vector de Fiedler"), (ax[1], grupo_exacto, "mínimo exacto")]:
    nx.draw(F, pos_F, ax=a, with_labels=True, font_size=7,
            node_color=["seagreen" if f in grupo else "orange" for f in F])
    a.set_title(f"{titulo}: RatioCut = {ratiocut(F, grupo):.2f}")
plt.show()